In [1]:
import tensorflow as tf
import rasterio
import numpy as np

def load_sar_tiff_tf(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        # Fixed-range normalization
        vv = np.clip(vv, -35.0, 5.0)
        vh = np.clip(vh, -40.0, 0.0)

        vv = (vv + 35.0) / 40.0
        vh = (vh + 40.0) / 40.0

        tiles = []

        # 🔑 TILING: 2048 → 16 × (512×512)
        for i in range(0, 2048, 512):
            for j in range(0, 2048, 512):
                tile_vv = vv[i:i+512, j:j+512]
                tile_vh = vh[i:i+512, j:j+512]

                tile = np.stack([tile_vv, tile_vh], axis=-1)
                tiles.append(tile)

        return np.stack(tiles, axis=0).astype(np.float32)  # (16, 512, 512, 2)

    tiles = tf.numpy_function(_read, [path], tf.float32)
    tiles.set_shape([16, 512, 512, 2])
    return tiles

In [2]:
import os

DATASET_ROOT = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
IMAGES_ROOT = os.path.join(DATASET_ROOT, "Images")

OIL_DIR = os.path.join(IMAGES_ROOT, "Oil")
NO_OIL_DIR = os.path.join(IMAGES_ROOT, "No_Oil")

oil_files = [os.path.join(OIL_DIR, f) for f in os.listdir(OIL_DIR) if f.endswith(".tif")]
no_oil_files = [os.path.join(NO_OIL_DIR, f) for f in os.listdir(NO_OIL_DIR) if f.endswith(".tif")]

print("Oil images:", len(oil_files))
print("No-Oil images:", len(no_oil_files))

Oil images: 150
No-Oil images: 150


In [3]:
def make_tiled_dataset(paths, label):
    ds = tf.data.Dataset.from_tensor_slices(paths)

    ds = ds.map(
        lambda x: load_sar_tiff_tf(x),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 🔥 Flatten (16 tiles → 16 samples)
    ds = ds.flat_map(
        lambda tiles: tf.data.Dataset.from_tensor_slices(
            (tiles, tf.repeat(label, 16))
        )
    )

    return ds


In [4]:
oil_ds = make_tiled_dataset(oil_files, 1)
no_oil_ds = make_tiled_dataset(no_oil_files, 0)

dataset = oil_ds.concatenate(no_oil_ds)

total_tiles = (len(oil_files) + len(no_oil_files)) * 16
print("Total tiles:", total_tiles)

dataset = dataset.shuffle(
    buffer_size=total_tiles,
    reshuffle_each_iteration=False
)

dataset = dataset.cache()
dataset = dataset.prefetch(tf.data.AUTOTUNE)

options = tf.data.Options()
options.experimental_deterministic = False
dataset = dataset.with_options(options)


I0000 00:00:1767764912.088130 2543311 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1767764912.088275 2543311 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Total tiles: 4800


# Spilt

In [5]:
num_oil = len(oil_files)
num_no_oil = len(no_oil_files)

total_samples = num_oil + num_no_oil
print("Total samples:", total_samples)

Total samples: 300


In [6]:
dataset = dataset.shuffle(
    buffer_size=total_samples,
    reshuffle_each_iteration=False  # 🔑 important
)

In [7]:
train_size = int(0.8 * total_tiles)

train_ds = dataset.take(train_size)
test_ds  = dataset.skip(train_size)

BATCH_SIZE = 8

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [8]:
x, y = next(iter(train_ds))
print("Batch shape:", x.shape)

2026-01-07 11:23:17.792478: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Batch shape: (8, 512, 512, 2)


# Model

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def build_detection_cnn():
    model = Sequential()

    for i in range(6):
        if i == 0:
            model.add(Conv2D(
                32, (3, 3),
                activation='relu',
                padding='same',
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                32, (3, 3),
                activation='relu',
                padding='same'
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(20, activation='relu'))
    model.add(Dense(20, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [10]:
model = build_detection_cnn()
model.summary()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 512, 512, 32)   │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20)             │        40,980 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,269 (344.80 KB)

 Trainable params: 88,269 (344.80 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(train_ds, epochs=15)

Epoch 1/15
    477/Unknown 443s 324ms/step - accuracy: 0.5112 - loss: 0.7895

2026-01-07 11:30:42.491411: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


480/480 ━━━━━━━━━━━━━━━━━━━━ 445s 326ms/step - accuracy: 0.5116 - loss: 0.7890


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Epoch 2/15
478/480 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.6608 - loss: 0.6264

...

480/480 ━━━━━━━━━━━━━━━━━━━━ 395s 191ms/step - accuracy: 0.6831 - loss: 5.0572
Epoch 14/15
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.6929 - loss: 1.5751

2026-01-07 12:58:37.445044: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


480/480 ━━━━━━━━━━━━━━━━━━━━ 372s 172ms/step - accuracy: 0.6929 - loss: 1.5755
Epoch 15/15
480/480 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.6986 - loss: 3.1048

2026-01-07 13:04:42.662904: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


480/480 ━━━━━━━━━━━━━━━━━━━━ 365s 156ms/step - accuracy: 0.6986 - loss: 3.1040


In [12]:
train_loss, train_acc = model.evaluate(train_ds, verbose=0)
print(f"Training Accuracy: {train_acc * 100:.2f}%")

2026-01-07 13:10:47.603773: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Training Accuracy: 68.49%


In [13]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Testing Accuracy: {test_acc * 100:.2f}%")

Testing Accuracy: 69.90%


In [14]:
print("===================================")
print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Test  Accuracy: {test_acc * 100:.2f}%")
print("===================================")

Train Accuracy: 68.49%
Test  Accuracy: 69.90%
